# Milestone 3: Visualization & Exploratory Analysis
**Project:** Public Health Data Visualization System  
**Dataset:** Global Health Statistics (1,000,000 records × 22 columns + engineered features from M2)  

**Objective:** Transform the cleaned and feature-engineered dataset from Milestone 2 into a visual discovery system that reveals health patterns across countries, time, and disease categories.

---
**Assigned to:** *(Abby, Asami, Annbel, Christine, Chrystabel, Esther, Joyce, Rita, Sharon)*  
**Branch:** `name/milestone3-task`

> **Before starting:** Run `load_dataset.ipynb` first, then run the Setup cell below.

---
## Collaboration & Pairing Notes

| Pair | M3 Section | M4 Section | How They Connect |
|---|---|---|---|
| **Pair 1 — Trend & Map Specialists** | Section 3: Spatial & Temporal Exploration | Section 8: Trend Analysis & Uncertainty | M3 shows *where/when* trends happen visually; M4 models those trends and calculates margin of error |
| **Pair 2 — Core Relationships Team** | Section 2: Distribution & Relationship Mapping | Section 6: Correlation & Regression | M3 shows the visual scatter cloud; M4 calculates the line of best fit and R-squared |
| **Pair 3 — Design & Logic Validators** | Section 1: Visual Strategy & Comparative Design | Section 5: Hypothesis Testing & Significance | M3 compares two groups visually; M4 runs the T-test to prove the difference is statistically significant |
| **Pair 4 — System Foundation Duo** | Section 4: Design Justification & Insight Reporting | Section 7: Data Splitting & Predictive Validation | M3 writes the final insight justification; M4 ensures models are honest via train/test split |
| **Solo — Quality Controller** | Evolutionary Failure Log | Evolutionary Failure Log | Interviews all pairs: *What chart didn't work? Which model failed?* |

> **Rule:** Every section must use `df` loaded from `data/processed/global_health_enriched.csv.gz`. Do not reload the raw CSV.

---
## Setup & Data Loading
**Owned by:** All members — run this first before any section.

**What to do here:**
- Load the processed dataset from Milestone 2 (`global_health_enriched.csv.gz`)
- Import all visualization libraries (`matplotlib`, `seaborn`, `plotly`, etc.)
- Confirm all M2 engineered features are present: `Severity_Index`, `DALY_Intensity`, `Avg_Incidence_Disease`, `High_Risk_Demographic`, `Vaccine_Available_Flag`, `Mortality_YoY_Change`, `Weighted_Time_Impact`, `Demographic_encoded`, `decade`, `Gender_Encoded`, `Disease_Category_Encoded`
- Define the shared `PALETTE` dictionary used across all charts in this notebook
- Set global `plt.rcParams` for consistent figure styling

**Deliverable check:** `df` loaded here is the single source of truth. All sections read from this variable only.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path

# ── Load M2 processed output ──────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'global_health_enriched.csv.gz'

df = pd.read_csv(PROCESSED_PATH, compression='gzip')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# ── Confirm M2 engineered features are present ────────────────────────────────
M2_FEATURES = [
    'Severity_Index', 'DALY_Intensity', 'Avg_Incidence_Disease',
    'High_Risk_Demographic', 'Vaccine_Available_Flag', 'Mortality_YoY_Change',
    'Weighted_Time_Impact', 'Demographic_encoded', 'decade',
    'Gender_Encoded', 'Disease_Category_Encoded'
]
missing = [f for f in M2_FEATURES if f not in df.columns]
if missing:
    
    print("=" * 150)
    print(f'WARNING — Missing M2 features: {missing}')
    print('Re-run milestone2_data_processing_transformation.ipynb first.')
    print("=" * 150)
else:
    print('\n ✓ All M2 engineered features confirmed.')
    print("=" * 150)

# ── Shared colour palette (colour-blind safe) ─────────────────────────────────
PALETTE = {
    'primary':    "#3FA8FF",
    'secondary':  "#FF9B06",
    'danger':     "#FC00EB",
    'success':    "#04B00A",
    'neutral':    "#898787",
    'sequential': 'YlOrRd',
    'diverging':  'RdYlGn',
    'categorical': 'tab10'
}

# ── Global figure styling ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

# ── Charts output directory ───────────────────────────────────────────────────
CHARTS_DIR = PROJECT_ROOT / 'docs' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

df.head(3)

Loaded: 1,000,000 rows × 33 columns

 ✓ All M2 engineered features confirmed.


,Country,Year,Disease Name,Disease Category,Prevalence Rate (%),Incidence Rate (%),Mortality Rate (%),Age Group,Gender,Population Affected,...,DALY_Intensity,Vaccine_Available_Flag,High_Risk_Demographic,Avg_Incidence_Disease,Mortality_YoY_Change,Weighted_Time_Impact,decade,Demographic_encoded,Gender_Encoded,Disease_Category_Encoded
0,argentina,2000,alzheimer's disease,respiratory,7.06,13.33,9.74,0-18,female,179830,...,0.007657,0,False,7.53279,NaN,0.0,2000,0,0,9
1,argentina,2000,alzheimer's disease,chronic,19.87,4.06,5.53,19-35,female,677757,...,0.002306,1,False,7.53279,-4.21,0.0,2000,1,0,3
2,argentina,2000,alzheimer's disease,metabolic,3.08,1.81,8.22,61+,male,100197,...,0.027626,1,True,7.53279,2.69,0.0,2000,3,1,6


---
## 1. Visual Strategy & Comparative Design
**Owned by:** Pair 3 — Design & Logic Validators  
**M4 Dependency → Section 5 (Hypothesis Testing):** The two groups you visually separate here (e.g., high-mortality vs. low-mortality countries) become the exact groups tested with a T-test in M4 Section 5.

**What to do here:**
- Choose one core metric: `Mortality Rate (%)`, `Healthcare Access (%)`, or `Recovery Rate (%)`
- Aggregate the chosen metric by `Country` using the mean
- Create **Chart A — Global Choropleth Heatmap**: geographic distribution of the metric across all countries
- Create **Chart B — Ranked Bar Chart**: top 10 and/or bottom 10 countries for the same metric
- Use the shared `PALETTE` from Setup for both charts
- Optionally annotate bars with `Severity_Index` (M2 feature) to add a second dimension
- Save both charts to `docs/charts/`

**Deliverable check:** Both charts must use the same aggregated data. Fill in the comparison table below after running the charts.

In [6]:
# Section 1: Chart A — Global Choropleth Heatmap


In [7]:
# Section 1: Chart B — Ranked Bar Chart (Top / Bottom N Countries)


### Comparison Documentation — Fill in after running the charts above

| Question | Your Answer |
|---|---|
| Which chart makes geographic outliers easiest to spot? | |
| Which chart makes relative country rankings easiest to compare? | |
| Which chart better supports the selected health question? | |
| Which two groups will you hand to M4 Section 5 for T-testing? | |

---
## 2. Core Distribution & Relationship Mapping
**Owned by:** Pair 2 — Core Relationships Team  
**M4 Dependency → Section 6 (Correlation & Regression):** The variables whose relationship you spot visually here become the inputs to the correlation matrix and regression model in M4 Section 6.

**What to do here:**
- Build a **Histogram** of `Mortality Rate (%)` — show the overall distribution shape with 40 bins
- Overlay the mean as a vertical reference line; optionally annotate with mean `Severity_Index` (M2 feature)
- Build a **Scatter Plot** of `Per Capita Income (USD)` vs. `Healthcare Access (%)` — colour points by `Disease Category`
- Add a **regression trend line** to the scatter plot
- Place both charts in a `1 × 2` subplot layout matching the M1 visual style
- Use colour-blind-safe palettes, clear axis labels with units, and a high data-ink ratio
- Save the figure to `docs/charts/`

**Deliverable check:** Both charts must be readable in greyscale. Axes must include units. No default matplotlib titles.

In [8]:
# Section 2: Chart A — Histogram of Mortality Rate (%)


In [9]:
# Section 2: Chart B — Scatter Plot: Per Capita Income vs. Healthcare Access (%)


---
## 3. Spatial & Temporal Exploration
**Owned by:** Pair 1 — Trend & Map Specialists  
**M4 Dependency → Section 8 (Trend Analysis & Uncertainty):** The yearly trend you plot here becomes the time-series you model and forecast in M4 Section 8. The visual trend line you draw here is what M4 will calculate a margin of error for.

**What to do here:**
- Build a **Time-Series Line Chart** of average `Mortality Rate (%)` by `Year` — one line per `Disease Category`
- Use the `decade` column (M2 engineered feature) to annotate decade boundaries on the time-series
- Optionally use `Mortality_YoY_Change` (M2 engineered feature) as a colour overlay to highlight year-on-year shifts
- Build a **Choropleth Map** of `Healthcare Access (%)` or `Mortality Rate (%)` for a single selected year
- Use year-level aggregation for the time-series and country-level aggregation for the map
- Save both outputs to `docs/charts/`

**Deliverable check:** Time-series x-axis must be labelled `Year`; y-axis must include the metric name and unit. The map must have a colour scale legend.

In [10]:
# Section 3: Chart A — Time-Series: Mortality Rate by Year and Disease Category


In [11]:
# Section 3: Chart B — Choropleth Map for a Selected Year


---
## 4. Design Justification & Insight Reporting
**Owned by:** Pair 4 — System Foundation Duo  
**M4 Dependency → Section 7 (Data Splitting & Predictive Validation):** The insights you document here must survive predictive validation in M4 Section 7. Any insight that the train/test model cannot support must be moved to the Evolutionary Failure Log.

**What to do here:**
- Review all charts from Sections 1–3 and identify at least **5 health insights**
- For each insight, state which chart it came from and which M2 engineered feature (if any) made it visible
- Justify every visual encoding choice: why this chart type, this colour scale, this axis range?
- Flag which insights need a statistical test in M4 to confirm or reject them

**Deliverable check:** Every insight must reference a specific chart. No generic observations allowed.

### Insight Reporting — Fill in after completing Sections 1–3

| # | Insight | Source Chart | M2 Feature Used | Needs M4 Validation? |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

### Design Justification — Fill in after completing Sections 1–3

| Chart | Encoding Choice | Justification |
|---|---|---|
| Choropleth (Section 1) | | |
| Bar chart (Section 1) | | |
| Histogram (Section 2) | | |
| Scatter plot (Section 2) | | |
| Time-series (Section 3) | | |

---
## Evolutionary Failure Log
**Owned by:** Solo — Quality Controller  
**Responsibility:** Interview all four pairs after they complete their sections. Ask: *"What chart didn't work? What looked like a pattern but turned out to be noise?"*

**What to document here:**
- Which visual pattern looked significant but was likely noise?
- Which chart was misleading or hard to interpret and why?
- Which M3 visualization needs a statistical test in M4 to confirm or reject it?
- Which M2 engineered feature did not add visual value and should be reconsidered?

This log is **non-negotiable** — it is the bridge between M3 visual discovery and M4 statistical validation.

### Failure Log — Fill in after all sections are complete

| # | What Failed / Looked Misleading | Section | Reason / Hypothesis | Action in M4 |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |